In [ ]:
import numpy as np
import potcorr
import utli
import lab

import matplotlib.pyplot as plt
import const

In [ ]:
import matplotlib as mpl

from scipy.interpolate import interp1d

# 生成一些数据
data = np.random.randn(30, 30)

# 选择一个预设的colormap
original_cmap = plt.cm.PuOr_r

# 定义新旧colormap之间的非线性映射
new_positions = [0, 0.5, 1]  # 新colormap中的位置
old_positions = [0.25, 0.5, 0.75]  # 旧colormap中对应的位置
mapping_function = interp1d(new_positions, old_positions)

# 使用映射函数生成自定义colormap的颜色列表
custom_colors = original_cmap(mapping_function(np.linspace(0, 1, 256)))

# 创建自定义colormap
custom_cmap = mpl.colors.LinearSegmentedColormap.from_list('custom_cmap', custom_colors)

# 创建图表并应用自定义colormap
plt.imshow(data, cmap=custom_cmap)
plt.colorbar()
plt.show()

In [ ]:
cell=lab.mos2_9x9
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
ttkw.get_vcoul()

In [ ]:
rho_6x6_ext = np.load('/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext_cut60.npy')


In [ ]:
pot_LR = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/pot_tot_LR.npy")
pot_dft = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/pot_tot_dft.npy")

In [ ]:
pot_LR2 = np.zeros_like(pot_LR)
pot_LR2[:135, :135,:]=pot_LR[135:, 135:, :]
pot_LR2[-135:, :135,:]=pot_LR[:135, 135:, :]
pot_LR2[-135:, -135:,:]=pot_LR[:135, :135, :]
pot_LR2[:135, -135:,:]=pot_LR[135:, :135, :]

pot_dft2 = np.zeros_like(pot_dft)
pot_dft2[:135, :135,:]=pot_dft[135:, 135:, :]
pot_dft2[-135:, :135,:]=pot_dft[:135, 135:, :]
pot_dft2[-135:, -135:,:]=pot_dft[:135, :135, :]
pot_dft2[:135, -135:,:]=pot_dft[135:, :135, :]

In [ ]:
rho_ext_6to9 = np.zeros_like(ttkw.v_coul)

In [ ]:
rho_ext_6to9[:90, :90,:]=rho_6x6_ext[90:, 90:, :]
rho_ext_6to9[-90:, :90,:]=rho_6x6_ext[:90, 90:, :]
rho_ext_6to9[-90:, -90:,:]=rho_6x6_ext[:90, :90, :]
rho_ext_6to9[:90, -90:,:]=rho_6x6_ext[90:, :90, :]

In [ ]:
np.sum(rho_ext_6to9)*ttkw.omega/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz

In [ ]:
#pot_dn = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/dn/Pot.dat")
#pot_dc = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/dc/Pot.dat")

pot_dv_nscs_hart = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/dc/dV_nscs-hart.dat")
pot_V_sr = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/dc/V_cd-sr.dat")
#pot_tot = pot_dc-pot_dn

#pot_dn = utli.read_dat("/anvil/scratch/x-rg47749/Si/666-P/dn/Pot.dat")
#pot_dc = utli.read_dat("/anvil/scratch/x-rg47749/Si/666-P/dc+1/Pot.dat")
#pot_tot = pot_dc-pot_dn

#pot_d = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/d-relax/Pot.dat")
#pot_p = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/p/Pot.dat")

#rho_dn = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/dn/Rho.dat")
#rho_dc = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/dc/Rho.dat")

#rho_dn = utli.read_dat("/anvil/scratch/x-rg47749/Si/666-P/dn/Rho.dat")
#rho_dc = utli.read_dat("/anvil/scratch/x-rg47749/Si/666-P/dc+1/Rho.dat")
#rho_tot = rho_dc-rho_dn

In [ ]:
Bohr_R = 0.52917721067




def dat2xsf(dat_inp, xsf_outp,rho_):
    file = open(dat_inp)
    f = file.readlines()

    nx, ny, nz = int(f[1].split()[0]), int(f[1].split()[1]), int(f[1].split()[2])
    natom, ntype = int(f[1].split()[6]),int(f[1].split()[7])
    alat = float(f[2].split()[1])
    a1 = Bohr_R*alat*np.array([float(f[3].split()[0]),float(f[3].split()[1]),float(f[3].split()[2])])
    a2 = Bohr_R*alat*np.array([float(f[4].split()[0]),float(f[4].split()[1]),float(f[4].split()[2])])
    a3 = Bohr_R*alat*np.array([float(f[5].split()[0]),float(f[5].split()[1]),float(f[5].split()[2])])
    Nheader = 7+natom+ntype
    vl = ''.join(f[Nheader:])
    vn = [float(i) for i in vl.split()]
    __rho__ = np.array(vn).reshape([nz,ny,nx])
    rho = __rho__.transpose(2,1,0) 
    rho = rho_


    with open(xsf_outp, 'w') as v_file:
        print('Crystal\n PRIMVEC\n', end=' ',file=v_file)
        print("%16.10f %16.10f %16.10f\n%16.10f %16.10f %16.10f\n%16.10f %16.10f %16.10f\n" 
            % (float(a1[0]), float(a1[1]), float(a1[2]),
                float(a2[0]), float(a2[1]), float(a2[2]), 
                float(a3[0]), float(a3[1]), float(a3[2])),
                end='', file=v_file)
        print(' PRIMCOORD\n 0    1\n', end=' ',file=v_file)
        print(' BEGIN_BLOCK_DATAGRID_3D\n 3D_PWSCF\n BEGIN_DATAGRID_3D_UNKNOWN\n', end=' ',file=v_file)
        print("%8d %8d %8d\n" % (nx, ny, nz), end=' ', file=v_file)
        print(' 0.00 0.00 0.00\n', end=' ',file=v_file)
        print("%16.10f %16.10f %16.10f\n%16.10f %16.10f %16.10f\n%16.10f %16.10f %16.10f\n" 
            % (float(a1[0]), float(a1[1]), float(a1[2]),
                float(a2[0]), float(a2[1]), float(a2[2]), 
                float(a3[0]), float(a3[1]), float(a3[2])),
                end='', file=v_file)

        count = 0
        for k in range(nz):
            for j in range(ny):
                for i in range(nx):
                    print("%16.10f" % rho[i,j,k], end='', file=v_file)
                    count+=1
                    if count%5==0:
                        print(end='\n', file=v_file)
        if (nx*ny*nz)%5 != 0:
            print(end='\n', file=v_file)
        
        print(' END_DATAGRID_3D\n', end=' ',file=v_file)
        print(' END_BLOCK_DATAGRID_3D', end=' ',file=v_file)


fname = '/anvil/scratch/x-rg47749/Si/666-P/dc+1/Rho.dat'
file_dir = '/anvil/scratch/x-rg47749/Si/666-P/dc+1/rho_ext.xsf'
dat2xsf(fname, file_dir,rho_ext.real)

In [ ]:
rho_ext.shape

In [ ]:
pot_jm = np.ones_like(pot_tot) 

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

#x_plot = (0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]+0.5*ttkw.fft_zz[:,:,ttkw.fft_nz//2]).flatten()
#y_plot = (0.5*ttkw.fft_xx[:,:,ttkw.fft_nz//2]+0.5*ttkw.fft_zz[:,:,ttkw.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)


#pot_k = np.fft.fftn(rho_tot)*ttkw.v_coul

#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=rho_ext_6to9[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
                #vmax=9e-7,
                #vmin=-0e-5,
                    #cmap='coolwarm')
                 cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

#x_plot = (0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]+0.5*ttkw.fft_zz[:,:,ttkw.fft_nz//2]).flatten()
#y_plot = (0.5*ttkw.fft_xx[:,:,ttkw.fft_nz//2]+0.5*ttkw.fft_zz[:,:,ttkw.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pot_dv_nscs_hart.real[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
                #vmax=2,
                #vmin=-1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
ttkw.lattpara

In [ ]:
plt.figure(figsize=(8, 6))
cp = plt.tricontourf(x_plot, y_plot, pot_dv_nscs_hart[:,:,ttkw.fft_nz//2].flatten(), 50, cmap='coolwarm')  # 20 控制等高线的数量

# 在颜色图上添加等高线
#plt.tricontour(X, Y, Z, 5, colors='black', linewidths=0.5)

# 添加颜色条
plt.colorbar(cp)
plt.axis('equal')
# 添加标题和轴标签
plt.title('2D Colormap with Contour Lines')
plt.xlabel('X axis')
plt.ylabel('Y axis')

plt.show()

In [ ]:
plt.plot(rho_ext[:,0,0])
plt.plot(rho_tot[:,0,0])
plt.plot(rho_induced_r[:,0,0])

In [ ]:
plt.plot(rho_ext[:,108,108])
plt.plot(rho_tot[:,108,108])
#plt.plot(rho_induced_r[:,108,108])

In [ ]:
plt.plot(rho_ext[:,108,108])
plt.plot(rho_tot[:,108,108])

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(rho_ext_6)[:,:,ttkw.fft_nz//2+0], 
                #gridsize=100,
                vmax=0.01,
                vmin=-0.01,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(rho_tot-1*rho_ind)[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
                vmax=0.01,
                vmin=-0.01,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_induced_6 =  rho_tot-rho_ext_6

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_15)[:,:,225//2-10:225//2+10], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.005,
                vmin=-0.005,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_15_m)[:,:,225//2-10:225//2+10], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.005,
                vmin=-0.005,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
3.186/0.529*12/6

In [ ]:

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(np.fft.ifftn(ttkw.v_coul2d))[:,:,ttkw.fft_nz-1],marker='H', s=4,
                #gridsize=100,
                vmax=0.001,
                vmin=-0.001,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(np.fft.ifftn(ttkw.v_coul2d))[:,:,ttkw.fft_nz//2],marker='H', s=4,
                #gridsize=100,
                vmax=0.001,
                vmin=-0.001,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
#plt.plot((rho_tot-rho_ext_18)[ttkw.fft_nx//2,:,ttkw.fft_nz//2].real)
#plt.plot((rho_tot)[ttkw.fft_nx//2,:,ttkw.fft_nz//2].real)
plt.plot((rho_ext_12)[ttkw.fft_nx//2,:,ttkw.fft_nz//2].real)
plt.plot((rho_ext_12_m)[ttkw.fft_nx//2,:,ttkw.fft_nz//2].real)
plt.ylim(-0.001, 0.001)

In [ ]:
dist_arr_tot = utli.distance_array_2d(ttkw.fft_nx, ttkw.fft_ny,a = ttkw.lattpara[0]*const.Bohr_R,b=ttkw.lattpara[1]*const.Bohr_R, defect_loc='center')

fig, ax = plt.subplots(figsize=(6,3),dpi=300)

for i in range(1):
    ax.scatter(dist_arr_tot.flatten(), rho_tot[:,:,ttkw.fft_nz//2+i].flatten().real,alpha=1, c='#fdb863',s=30,label=r'$\Delta\rho_{tot}$')
    ax.scatter(dist_arr_tot.flatten(), rho_6x6_ext[:,:,ttkw.fft_nz//2+i].flatten().real,alpha=1 , c='#B2ABD2',s=30,label=r'$\Delta\rho_{ex}$')
    
plt.xlabel(r'Distance from Defect ($\AA$)')
plt.ylabel('Charge density')
plt.legend()
plt.show()

In [ ]:
dist_arr_tot = utli.distance_array_2d(ttkw.fft_nx, ttkw.fft_ny,a = ttkw.lattpara[0]*const.Bohr_R,b=ttkw.lattpara[1]*const.Bohr_R, defect_loc=[0,0,0.5])

fig, ax = plt.subplots(figsize=(6,3),dpi=300)

for i in range(1):
    ax.scatter(dist_arr_tot.flatten(), 13.6*pot_dft[:,:,ttkw.fft_nz//2+i].flatten().real,alpha=1, c='#fdb863',s=30,label=r'$\Delta V_{DFT}$')
    ax.scatter(dist_arr_tot.flatten(), 13.6*pot_LR[:,:,ttkw.fft_nz//2+i].flatten().real,alpha=1 , c='#B2ABD2',s=30,label=r'$\Delta V_{LR}$')
    
plt.xlabel('Distance from Defect (A)')
plt.ylabel('Potential (eV)')
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,3),dpi=300)
for i in range(1):
    plt.plot(np.arange(270)*3.186*9/270,pot_dft[135+i,:,225//2+10],label='DFT',c='red')
    plt.plot(np.arange(270)*3.186*9/270,(pot_LR+0.00151364305)[135+i,:,225//2+10],label='LR',c='blue')
#plt.plot(np.arange(225)*3.186*9/270,pot_dft[135,135,:])
#plt.plot(np.arange(225)*3.186*9/270,(pot_LR+0.00131364305)[135,135,:])
plt.xlabel('x ($\\mathrm{\\AA}$)')
plt.ylabel(r'Potential ($\mathrm{\AA}^{-3}$)')
plt.legend()
plt.show()

In [ ]:
np.sum(pot_LR+0.00151364305)

In [ ]:
np.sum(pot_dft)

In [ ]:
pot_dft.shape

In [ ]:
rho_ind = rho_tot - rho_ext_12
print(np.sum(rho_ind))
print(np.sum(rho_tot))

In [ ]:
dist_arr_tot = utli.distance_array_2d(ttkw.fft_nx, ttkw.fft_ny,a = ttkw.lattpara[0],b=ttkw.lattpara[1], defect_loc='center')

fig, ax = plt.subplots(figsize=(12,4),dpi=100)

#ax.scatter(dist_arr_tot.flatten(), 2*rho_ind[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.05 )
ax.scatter(dist_arr_tot.flatten(), (rho_ext_12)[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.5 )
ax.scatter(dist_arr_tot.flatten(), (rho_tot-2*rho_ind)[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.5 )

#ax.scatter(dist_arr_tot.flatten(), rho_ext_12_m[:,:,ttkw.fft_nz//2].flatten().real,alpha=0.4)

plt.xlabel('Distance from Defect (Bohr)')
plt.ylabel('Potential (Ry)')

plt.ylim(-0.0025,0.0025)
plt.legend()
plt.show()

In [ ]:
rho_tot_p = np.copy(rho_tot)
rho_tot_p[rho_tot_p<0]=0

rho_tot_n = np.copy(rho_tot)
rho_tot_n[rho_tot_n>0]=0

rho_ext_p = np.copy(rho_ext)
rho_ext_p[rho_ext_p<0]=0

rho_ext_n = np.copy(rho_ext)
rho_ext_n[rho_ext_n>0]=0

rho_dl_p = np.copy(0.5*(rho_B466+rho_B467))
rho_dl_p[rho_ext_p<0]=0

rho_dl_n = np.copy(0.5*(rho_B466+rho_B467))
rho_dl_n[rho_ext_n>0]=0

rho_ind_p = np.copy(rho_tot-rho_ext)
rho_ind_p[rho_ext_p<0]=0

rho_ind_n = np.copy(rho_tot-rho_ext)
rho_ind_n[rho_ext_n>0]=0

fig, ax = plt.subplots(figsize=(9,6),dpi=600)
#plt.plot(ttkw.fft_x,np.sum(rho_tot_p[:,:,225//2], axis=(0)))
#plt.plot(ttkw.fft_x,np.sum(-rho_tot_n[:,:,225//2], axis=(0)))
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(rho_tot_p[:,:,225//2-5:225//2+5], axis=(1,2)), color="r", alpha=0.4)
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(-rho_tot_n[:,:,225//2-5:225//2+5], axis=(1,2)), color="b", alpha=0.4)
plt.ylim(0,3)
plt.show()

fig, ax = plt.subplots(figsize=(9,6),dpi=600)
#plt.plot(ttkw.fft_x,np.sum(rho_ext_p[:,:,225//2], axis=(0)))
#plt.plot(ttkw.fft_x,np.sum(-rho_ext_n[:,:,225//2], axis=(0)))
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(rho_ext_p[:,:,225//2-5:225//2+5], axis=(1,2)), color="r", alpha=0.4)
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(-rho_ext_n[:,:,225//2-5:225//2+5], axis=(1,2)), color="b", alpha=0.4)
plt.ylim(0,3)
plt.show()

fig, ax = plt.subplots(figsize=(9,6),dpi=600)
#plt.plot(ttkw.fft_x,np.sum(rho_ext_p[:,:,225//2], axis=(0)))
#plt.plot(ttkw.fft_x,np.sum(-rho_ext_n[:,:,225//2], axis=(0)))
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(rho_dl_p[:,:,225//2-5:225//2+5], axis=(1,2)), color="r", alpha=0.4)
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(-rho_dl_n[:,:,225//2-5:225//2+5], axis=(1,2)), color="b", alpha=0.4)
plt.ylim(0,3)
plt.show()

fig, ax = plt.subplots(figsize=(9,6),dpi=600)
#plt.plot(ttkw.fft_x,np.sum(rho_ext_p[:,:,225//2], axis=(0)))
#plt.plot(ttkw.fft_x,np.sum(-rho_ext_n[:,:,225//2], axis=(0)))
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(rho_ind_p[:,:,225//2-5:225//2+5], axis=(1,2)), color="r", alpha=0.4)
plt.fill_between(ttkw.fft_x*np.sqrt(3)/2, np.sum(-rho_ind_n[:,:,225//2-5:225//2+5], axis=(1,2)), color="b", alpha=0.4)
plt.ylim(0,3)
plt.show()

In [ ]:
rho_ext = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext_cut60.npy")

In [ ]:
eps1 = '/anvil/scratch/x-rg47749/mos2/bgw_results/9x9/epsmat.h5'
eps0 = '/anvil/scratch/x-rg47749/mos2/bgw_results/12x12/eps0mat.h5'
#eps1 = '/anvil/projects/x-che190065/rjguo/Si/dielectric/666/chimat.h5'
#eps0 = '/anvil/projects/x-che190065/rjguo/Si/dielectric/666/chi0mat.h5'
ttkw.read_epsinv(eps1=eps1, eps0=eps0)

In [ ]:
ttkw.Eps1.nmtx

In [ ]:
file_path = '/anvil/projects/x-che190065/rjguo/Si/dielectric/444/kgrid.log'

# 打开文件并逐行读取内容
with open(file_path, 'r') as file:
    lines = file.readlines()

# 查找各部分的起始索引
#k_points_start_idx = None
symmetries_start_idx = None
#k_points_reduced_start_idx = None

for idx, line in enumerate(lines):
#    if "k-points in the original uniform grid" in line:
#        k_points_start_idx = idx + 2  # 跳过标题行和列名行
    if "symmetries of the crystal with FFT grid" in line:
        print('Find symmetries of the crystal with FFT grid')
        symmetries_start_idx = idx + 3  # 跳过标题行
#    if "k-points in the irreducible wedge" in line:
#        print('Find k-points in the irreducible wedge')
#        k_points_reduced_start_idx = idx + 2

# 确保找到了两部分
#if k_points_start_idx is None:
#    raise ValueError("No 'k-points in the original uniform grid' part")
#if k_points_reduced_start_idx is None:
#    raise ValueError("No 'k-points in the irreducible wedge' part")
if symmetries_start_idx is None:
    raise ValueError("No 'symmetries of the crystal with FFT grid' part")

# 读取k点数据
#k_points_data = []
#for line in lines[k_points_start_idx:]:
#    stripped_line = line.strip()
#    if stripped_line == '' or stripped_line.startswith('---'):
#        continue
#    parts = stripped_line.split()
#    if len(parts) != 7:
#        break
#    k_points_data.append(parts)
    
#k_points_reduced_data = []
#for line in lines[k_points_reduced_start_idx:]:
#    stripped_line = line.strip()
#    if stripped_line == '' or stripped_line.startswith('---'):
#        continue
#    parts = stripped_line.split()
#    if len(parts) != 6:
#        break
#    k_points_reduced_data.append(parts)

# 读取对称性数据
symmetries_data = []
for line in lines[symmetries_start_idx:]:
    #print(line)
    stripped_line = line.strip()
    if stripped_line == '' or stripped_line.startswith('---'):
        continue
    parts = stripped_line.split()
    if len(parts) != 14:
        break
    symmetries_data.append(parts)

# 将k点数据转换为Pandas DataFrame
#k_points_columns = ['Index', 'kx', 'ky', 'kz', 'Weight', 'eq_kp', 'symop']
#k_points_df = pd.DataFrame(k_points_data, columns=k_points_columns)

# 将字符串列转换为合适的数据类型
#k_points_df['Index'] = k_points_df['Index'].astype(int)
#k_points_df['kx'] = k_points_df['kx'].astype(float)
#k_points_df['ky'] = k_points_df['ky'].astype(float)
#k_points_df['kz'] = k_points_df['kz'].astype(float)
#k_points_df['Weight'] = k_points_df['Weight'].astype(float)
#k_points_df['eq_kp'] = k_points_df['eq_kp'].astype(int)


#k_points_reduced_columns = ['Index', 'kx', 'ky', 'kz', 'Weight', 'Original idx']
#k_points_reduced_df = pd.DataFrame(k_points_reduced_data, columns=k_points_reduced_columns)
#k_points_reduced_df['Index'] = k_points_reduced_df['Index'].astype(int)
#k_points_reduced_df['kx'] = k_points_reduced_df['kx'].astype(float)
#k_points_reduced_df['ky'] = k_points_reduced_df['ky'].astype(float)
#k_points_reduced_df['kz'] = k_points_reduced_df['kz'].astype(float)
#k_points_reduced_df['Weight'] = k_points_reduced_df['Weight'].astype(float)
#k_points_reduced_df['Original idx'] = k_points_reduced_df['Original idx'].astype(int)


symmetries_dict = {}
for sym_data in symmetries_data:
    name = sym_data[0]
    matrix = np.array(sym_data[2:11], dtype=int).reshape(3, 3)
    symmetries_dict[name] = matrix
    
#symmetries_dict['---'] = np.array([[1,0,0],[0,1,0],[0,0,1]])



# 打印对称性字典以验证结果
for name, matrix in symmetries_dict.items():
    print(f"{name}:")
    print(matrix)
    print()


In [ ]:
x='01'
symmetries_dict['r'+str(x)]

In [ ]:
#k_symmetry_map =  ttkw.get_k_symmetry_map_from_symdict(symmetries_dict)
k_symmetry_map =  ttkw.get_k_symmetry_map()
#epsym_dict = ttkw.get_epsym_dict(k_symmetry_map, ecut=12)
#epsym_dict = ttkw.read_epsym_dict('/anvil/scratch/x-rg47749/data/mos2/symm/epsym_dict.pkl')  #

In [ ]:
epsym_dict = ttkw.get_epsym_dict(k_symmetry_map,ecut=45)

In [ ]:
epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict = ttkw.gen_mat_dict(k_symmetry_map, epsym_dict)

In [ ]:
import pickle

with open('/anvil/projects/x-che190065/rjguo/mos2/dielectric/symm/epsym_dict_6x6.pkl', 'wb') as f:
    pickle.dump(epsym_dict, f)

In [ ]:
#import pickle

with open('/anvil/projects/x-che190065/rjguo/mos2/dielectric/intrinsic/6x6/save/chimat_eps2rho_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_eps2rho_dict, f)
with open('/anvil/projects/x-che190065/rjguo/mos2/dielectric/intrinsic/6x6/save/chimat_eps2eps_irrbz_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_eps2eps_irrbz_dict, f)
with open('/anvil/projects/x-che190065/rjguo/mos2/dielectric/intrinsic/6x6/save/chimat_dict.pkl', 'wb') as f:
    pickle.dump(epsmat_dict, f)

In [ ]:
ttkw.vcoul2d0modify()
ttkw.vcoul0modify()

In [ ]:
#dpot_k = np.fft.fftn(rho_dd2 - rho_dd)*ttkw.v_coul2d
dpot_k = np.fft.fftn(rho_ext_6to9)*ttkw.v_coul

In [ ]:
rho_G = np.fft.fftn(rho_ext_6to9)

In [ ]:
wcoul_mat_dict = ttkw.gen_wcoul_mat_dict(k_symmetry_map,epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict )

In [ ]:
phi_G_dict = ttkw.gen_phi_G_dict(rho_G, k_symmetry_map, epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, wcoul_mat_dict)
phi_G = ttkw.map_phi_G(phi_G_dict, epsmat_eps2rho_dict)
phi_r = np.fft.ifftn(phi_G)

In [ ]:
rho_induced_G_dict = ttkw.gen_phi_G_dict(dpot_k, k_symmetry_map, epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict)

In [ ]:
rho_induced_k = ttkw.map_phi_G(rho_induced_G_dict, epsmat_eps2rho_dict)

In [ ]:
rho_induced_r = np.fft.ifftn(rho_induced_k).real

In [ ]:
rho_ext = rho_tot-rho_induced_r

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum(phi_r[:,:,270//2-1:270//2+1], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.12,
                vmin=0.03,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(4.709-8.16, 4.709+8.16)
#ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
pot_tot2 = np.fft.ifftn(np.fft.fftn(rho_tot)*ttkw.v_coul2d).real

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot[:,:,270//2-1:270//2+1], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.12,
                vmin=0.03,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(4.709-8.16, 4.709+8.16)
#ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
for i in range(15):
    plt.plot(phi_r[:,i, 225//2], c='orange')
    

for i in range(15):
    plt.plot(pot_tot[:,i, 225//2], c='blue')
    


In [ ]:
fig, ax = plt.subplots(figsize=(4,4),dpi=300)
xxx = (pot_LR+0.00151)[:,:, 225//2-10:225//2+3].flatten().real
yyy = pot_dft[:,:, 225//2-10:225//2+3].flatten().real
plt.plot(np.linspace(0.02,0.12,100), np.linspace(0.02,0.12,100), c='red', linestyle='--')
plt.scatter(xxx, yyy, s=20)

#plt.plot(np.arange(225)*3.186*9/270,(pot_LR+0.00131364305)[135,135,:])
plt.xlabel(r'DFT Potential (Ry)')
plt.ylabel(r'Predicted Potential (Ry')

plt.show()

In [ ]:
np.max((pot_LR[:,:,225//2:225//2+10] - (pot_dft+0.00151)[:,:,225//2:225//2+10]).real)

In [ ]:
np.save("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/pot_tot_LR.npy", phi_r)

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum((rho_tot-rho_ext)[:,:,225//2-5:225//2+5], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.03,
                vmin=-0.03,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(4.709-8.16, 4.709+8.16)
ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum(rho_ext[:,:,225//2-5:225//2+5], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.03,
                vmin=-0.03,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(4.709-8.16, 4.709+8.16)
ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()*const.Bohr_R
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()*const.Bohr_R
fig, ax = plt.subplots(figsize=(2,2),dpi=600)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=np.sum((0.5*(rho_B466+rho_B467))[:,:,225//2-5:225//2+5], axis=(2)).real, marker='h',s=1,
                #gridsize=100,
                vmax=0.03,
                vmin=-0.03,
                    cmap='bwr')
#plt.axhline(y=4.709*np.sqrt(3), color='r', linestyle='--', linewidth=0.7)
#plt.axvline(x=4.709, color='r', linestyle='--', linewidth=0.7)
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(4.709-8.16, 4.709+8.16)
ax.set_ylim(4.709*np.sqrt(3)-8.16, 4.709*np.sqrt(3)+8.16)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.025)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
plt.scatter(0.5*(rho_B1051+rho_B1052)[:,:,225//2-5:225//2+5].real,rho_ext[:,:,225//2-5:225//2+5].real.flatten(), c='purple')
plt.plot(range(-10,10), range(-10,10), c='red', linestyle='--')
plt.ylim(-0.01,0.04)
plt.xlim(-0.01,0.04)

In [ ]:
plt.plot(np.sum(rho_ext, axis=(2))[:,ttkw.fft_ny//2])
#plt.plot(np.sum(rho_induced_r, axis=(2))[:,ttkw.fft_ny//2])
plt.plot(np.sum(rho_tot, axis=(2))[:,ttkw.fft_ny//2])
#plt.plot(np.sum(rho_ext_6, axis=(2))[:,ttkw.fft_ny//2])
#plt.plot(np.sum(rho_ext_6[:,:,225//2], axis=(0)))
#plt.plot(np.sum(0.5*(rho_B466+rho_B467), axis=(2))[:,ttkw.fft_ny//2])

In [ ]:
plt.plot(np.sum(rho_ext, axis=(2))[:,0*ttkw.fft_ny//2])
#plt.plot(np.sum(rho_induced_r, axis=(2))[:,ttkw.fft_ny//2])
plt.plot(np.sum(rho_tot, axis=(2))[:,0*ttkw.fft_ny//2])
plt.plot(np.sum(0.5*(rho_B1051+rho_B1052), axis=(2))[:,0*ttkw.fft_ny//2])

In [ ]:
plt.figure(figsize=(8, 6))
cp = plt.tricontourf(x_plot, y_plot, rho_ext_18[:,:,ttkw.fft_nz//2].flatten(), 10, cmap='coolwarm')  # 20 控制等高线的数量

# 在颜色图上添加等高线
#plt.tricontour(X, Y, Z, 5, colors='black', linewidths=0.5)

# 添加颜色条
plt.colorbar(cp)
plt.axis('equal')
# 添加标题和轴标签
plt.title('2D Colormap with Contour Lines')
plt.xlabel('X axis')
plt.ylabel('Y axis')

plt.show()

In [ ]:
np.save("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/rho_ext_cut60.npy", rho_ext.real)

In [ ]:

plt.plot(np.linspace(-18,18, 540),np.sum(rho_ext_18_m, axis=(2,1)))
plt.plot(np.linspace(-15,15, 450),np.sum(rho_ext_15_m, axis=(2,1)))
plt.plot(np.linspace(-12,12, 360),np.sum(rho_ext_12_m, axis=(2,1)))
plt.plot(np.linspace(-9,9, 270),np.sum(rho_ext_9_m, axis=(2,1)))
plt.plot(np.linspace(-6,6, 180),np.sum(rho_ext_6_m, axis=(2,1)))

In [ ]:
#plt.plot(np.linspace(-10,10, 360),np.sum(rho_tot_12[:,:,:], axis=(2,1)))
plt.plot(np.linspace(-15,15, 450),np.sum(rho_ext[:,:,:], axis=(2,1)))

In [ ]:
plt.plot(np.linspace(-15,15, 540),np.sum(rho_ext_18[:,:,:], axis=(2,1)))
plt.plot(np.linspace(-15,15, 540),np.sum(rho_tot[:,:,:], axis=(2,1)))

In [ ]:
plt.plot(np.sum(rho_induced_r[:,:,:], axis=(0,1)))

In [ ]:
np.sum(rho_induced_r)

In [ ]:
rho_ext_18 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/rho_ext.npy")
rho_ext_15 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/rho_ext.npy")
rho_ext_12 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/rho_ext.npy")
rho_ext_9 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/rho_ext.npy")
rho_ext_6 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext.npy")

rho_ext_18_m = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/rho_ext_m.npy")
rho_ext_15_m = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/rho_ext_m.npy")
rho_ext_12_m = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/rho_ext_m.npy")
rho_ext_9_m = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/rho_ext_m.npy")
rho_ext_6_m = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext_m.npy")

In [ ]:
eps1 = '/anvil/projects/x-che190065/rjguo/mos2/dielectric/intrinsic/12x12/epsmat.h5'
eps0 = '/anvil/projects/x-che190065/rjguo/mos2/dielectric/intrinsic/12x12/eps0mat.h5'
ttkw.read_epsinv(eps1=eps1, eps0=eps0)

In [ ]:
k_symmetry_map =  ttkw.get_k_symmetry_map()
epsym_dict = ttkw.get_epsym_dict(k_symmetry_map, ecut=5)

In [ ]:
epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict = ttkw.gen_mat_dict(k_symmetry_map, epsym_dict)
wcoul_mat_dict = ttkw.gen_wcoul_mat_dict(k_symmetry_map,epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, epsmat_dict )

In [ ]:
rho_ext_k = np.fft.fftn(rho_ext_12)
phi_G_dict = ttkw.gen_phi_G_dict(rho_ext_k, k_symmetry_map, epsmat_eps2rho_dict, epsmat_eps2eps_irrbz_dict, wcoul_mat_dict)
phi_k = ttkw.map_phi_G(phi_G_dict, epsmat_eps2rho_dict)
phi_r = np.fft.ifftn(phi_k)

In [ ]:
plt.plot(phi_r[180,180,:])

In [ ]:
plt.plot(np.linspace(-6, 6, 180), rho_ext_6[:,90,225//2+8])
plt.plot(np.linspace(-9, 9, 270), rho_ext_9[:,135,225//2+8])
plt.plot(np.linspace(-12, 12, 360), rho_ext_12[:,180,225//2+8])
plt.plot(np.linspace(-15, 15, 450), rho_ext_15[:,225,225//2+8])
#plt.plot(np.linspace(-18, 18, 540), rho_ext_18[:,270,225//2])
#plt.plot(pot_tot[:,180,225//2])

In [ ]:
plt.plot(np.linspace(-6, 6, 180), rho_ext_6_m[:,90,225//2])
plt.plot(np.linspace(-9, 9, 270), rho_ext_9_m[:,135,225//2])
plt.plot(np.linspace(-12, 12, 360), rho_ext_12_m[:,180,225//2])
plt.plot(np.linspace(-15, 15, 450), rho_ext_15_m[:,225,225//2])
plt.plot(np.linspace(-18, 18, 540), rho_ext_18_m[:,270,225//2])
#plt.plot(pot_tot[:,180,225//2])
#plt.ylim(0.0075, 0.009)

In [ ]:
cell=lab.mos2_18x18
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
rho_ext_18 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/rho_ext.npy")
x_ = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])
y_ = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)

center_x = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])[ttkw.fft_nx//2, ttkw.fft_ny//2]
center_y = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)[ttkw.fft_nx//2, ttkw.fft_ny//2]
dist2center_plot = np.sqrt((x_-center_x)**2+(y_-center_y)**2)

rho_ext_18_m = np.copy(rho_ext_18)
rho_ext_18_m[dist2center_plot>(ttkw.lattpara[0]/9)]=0
print(np.sum(rho_ext_18_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
rho_ext_18_m = rho_ext_18_m/(np.sum(rho_ext_18_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
print(np.sum(rho_ext_18_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)


x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_18)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_18_m)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
cell=lab.mos2_15x15
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
rho_ext_15 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/rho_ext.npy")
x_ = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])
y_ = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)

rho_bare = np.zeros_like(rho_ext_15)

nx = ttkw.fft_nx
rho_bare[:nx//2,:nx//2,:]=rho_ext_15[nx//2:,nx//2:,:]
rho_bare[:nx//2,-nx//2:,:]=rho_ext_15[nx//2:,:nx//2,:]
rho_bare[-nx//2:,:nx//2,:]=rho_ext_15[:nx//2,-nx//2:,:]
rho_bare[-nx//2:,-nx//2:,:]=rho_ext_15[:nx//2,:nx//2,:]

rho_ext_15 = rho_bare

center_x = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])[ttkw.fft_nx//2, ttkw.fft_ny//2]
center_y = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)[ttkw.fft_nx//2, ttkw.fft_ny//2]
dist2center_plot = np.sqrt((x_-center_x)**2+(y_-center_y)**2)


rho_ext_15_m = np.copy(rho_ext_15)
rho_ext_15_m[dist2center_plot>(ttkw.lattpara[0]/7.5)]=0
print(np.sum(rho_ext_15_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
rho_ext_15_m = rho_ext_15_m/(np.sum(rho_ext_15_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
print(np.sum(rho_ext_15_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_15)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_15_m)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
cell=lab.mos2_12x12
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
rho_ext_12 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/rho_ext.npy").real
x_ = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])
y_ = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)

center_x = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])[ttkw.fft_nx//2, ttkw.fft_ny//2]
center_y = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)[ttkw.fft_nx//2, ttkw.fft_ny//2]
dist2center_plot = np.sqrt((x_-center_x)**2+(y_-center_y)**2)

rho_ext_12_m = np.copy(rho_ext_12)

rho_ext_12_m[dist2center_plot>(ttkw.lattpara[0]/4)]=0
print(np.sum(rho_ext_12_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
rho_ext_12_m = rho_ext_12_m/(np.sum(rho_ext_12_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
print(np.sum(rho_ext_12_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_12)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_12_m)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
cell=lab.mos2_9x9
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
rho_ext_9 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/rho_ext.npy")
x_ = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])
y_ = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)

rho_bare = np.zeros_like(rho_ext_9)

nx = ttkw.fft_nx
rho_bare[:nx//2,:nx//2,:]=rho_ext_9[nx//2:,nx//2:,:]
rho_bare[:nx//2,-nx//2:,:]=rho_ext_9[nx//2:,:nx//2,:]
rho_bare[-nx//2:,:nx//2,:]=rho_ext_9[:nx//2,-nx//2:,:]
rho_bare[-nx//2:,-nx//2:,:]=rho_ext_9[:nx//2,:nx//2,:]

rho_ext_9 = rho_bare

center_x = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])[ttkw.fft_nx//2, ttkw.fft_ny//2]
center_y = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)[ttkw.fft_nx//2, ttkw.fft_ny//2]
dist2center_plot = np.sqrt((x_-center_x)**2+(y_-center_y)**2)

rho_ext_9_m = np.copy(rho_ext_9)
rho_ext_9_m[dist2center_plot>(ttkw.lattpara[0]/4.5)]=0
print(np.sum(rho_ext_9_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
rho_ext_9_m = rho_ext_9_m/(np.sum(rho_ext_9_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
print(np.sum(rho_ext_9_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_9)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_9_m)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
cell=lab.mos2_6x6
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()
rho_ext_6 = np.load("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/rho_ext.npy")
x_ = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])
y_ = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)

center_x = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2])[ttkw.fft_nx//2, ttkw.fft_ny//2]
center_y = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2)[ttkw.fft_nx//2, ttkw.fft_ny//2]
dist2center_plot = np.sqrt((x_-center_x)**2+(y_-center_y)**2)

rho_ext_6_m = np.copy(rho_ext_6)
rho_ext_6_m[dist2center_plot>(ttkw.lattpara[0]/3)]=0
print(np.sum(rho_ext_6_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
rho_ext_6_m = rho_ext_6_m/(np.sum(rho_ext_6_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)
print(np.sum(rho_ext_6_m)/ttkw.fft_nx/ttkw.fft_ny/ttkw.fft_nz*ttkw.omega)

x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_6)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

fig, ax = plt.subplots(figsize=(9,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, 
                 c=np.sum((rho_ext_6_m)[:,:,225//2-50:225//2+50], axis=(2)).flatten(), 
                #gridsize=100,
                vmax=0.1,
                vmin=-0.1,
                    cmap='bwr',
                s=1,marker='H'
                )
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(-13, 50)
#ax.set_ylim(0, 63)
#cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_ext_6[np.abs(rho_ext_6)<3e-4]=0
rho_ext_12[np.abs(rho_ext_12)<3e-4]=0
rho_ext_18[np.abs(rho_ext_18)<3e-4]=0

In [ ]:
print(np.sum(rho_ext_6))
print(np.sum(rho_ext_12))
print(np.sum(rho_ext_18))

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=phi_r[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
                #vmax=2,
                #vmin=-1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_B1870 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/wfc_test/WFC.dat_K001_B1870")
rho_B1871 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/wfc_test/WFC.dat_K001_B1871")
rho_B1872 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/12x12/wfc_test/WFC.dat_K001_B1872")

In [ ]:
rho_B466 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/wfc_test/WFC.dat_K001_B466")
rho_B467 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/wfc_test/WFC.dat_K001_B467")
rho_B468 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/wfc_test/WFC.dat_K001_B468")

In [ ]:
rho_B1051 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/wfc_test/WFC.dat_K001_B1051")
rho_B1052 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/wfc_test/WFC.dat_K001_B1052")
rho_B1053 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/9x9/wfc_test/WFC.dat_K001_B1053")

In [ ]:
rho_B2923 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/wfc_test/WFC.dat_K001_B2923")
rho_B2924 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/wfc_test/WFC.dat_K001_B2924")
rho_B2925 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/15x15/wfc_test/WFC.dat_K001_B2925")

In [ ]:
rho_B4210 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/wfc_test/WFC.dat_K001_B4210")
rho_B4211 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/wfc_test/WFC.dat_K001_B4211")
rho_B4212 = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/18x18/wfc_test/WFC.dat_K001_B4212")

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=0.5*(rho_B4210 + rho_B4211)[:,:,ttkw.fft_nz//2], 
                #gridsize=100,
                vmax=0.001,
                vmin=-0.001,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
rho_ext_9_e = np.zeros([540,540,225])
rho_ext_9_e[540*1//4:540*2//4, 540//4:540*2//4, :] = rho_ext_9[270//2:, 270//2:, :]
rho_ext_9_e[540*2//4:540*3//4, 540//4:540*2//4, :] = rho_ext_9[:270//2, 270//2:, :]
rho_ext_9_e[540*2//4:540*3//4, 540*2//4:540*3//4, :] = rho_ext_9[:270//2, :270//2, :]
rho_ext_9_e[540*1//4:540*2//4, 540*2//4:540*3//4, :] = rho_ext_9[270//2:, :270//2, :]

In [ ]:
#rho_tot_6_e = np.zeros([180,180,225])
shift_units = 120
original_array = rho_tot
rolled_x = np.roll(original_array, shift_units, axis=0)
rho_dd = np.roll(rolled_x, -shift_units, axis=1)
#rho_dd = np.flip(rolled_xy, axis=2)
rho_dd2 = rho_tot
#rho_dd2 = np.flip(np.flip(rho_tot, axis=0), axis=1)

In [ ]:
x_plot = (ttkw.fft_xx[:,:,ttkw.fft_nz//2]-0.5*ttkw.fft_yy[:,:,ttkw.fft_nz//2]).flatten()
y_plot = (ttkw.fft_yy[:,:,ttkw.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(rho_dd2 - rho_dd)[:,:,225//2], 
                #gridsize=100,
                vmax=0.001,
                vmin=-0.001,
                    cmap='bwr')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
plt.plot(0.5*(rho_B466+rho_B467)[ttkw.fft_nx//2,:,ttkw.fft_nz//2-30].real)
plt.plot((rho_ext_6_m)[ttkw.fft_nx//2,:,ttkw.fft_nz//2-30].real)
plt.plot((rho_tot)[ttkw.fft_nx//2,:,ttkw.fft_nz//2-30].real)
#plt.ylim(-0.001, 0.001)

In [ ]:

#plt.plot(np.sum(1*(rho_B4210+rho_B4211), axis=(2))[540//2,:].real, label = 'neutral defect state', linewidth=2)
#plt.plot(np.sum(1*(rho_B2923+rho_B2924), axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
#plt.plot(np.sum(0.5*(rho_B1870+rho_B1871), axis=(2))[360//2,:].real, label = 'neutral defect state')
#plt.plot(np.sum(0.5*(rho_B1051+rho_B1052), axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(0.5*(rho_B466+rho_B467), axis=(2))[180//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(rho_ext, axis=(2))[180//2,:].real, label = 'neutral defect state')
#plt.plot(np.sum(rho_ext_18_m, axis=(2))[540//2,:].real, label = r"$\rho_{ext}$", linestyle='--')
#plt.plot(np.sum(rho_ext_18, axis=(2))[540//2,:].real, label = r"$\rho_{ext}$", linestyle='--')
#plt.plot(np.sum(rho_ext_15, axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
#plt.plot(np.sum(rho_ext_12 - 0.5*(rho_B1870+rho_B1871), axis=(2))[360//2,:].real, label = 'neutral defect state', linestyle='--')
#plt.plot(np.sum(rho_ext_9, axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
#plt.plot(np.sum(rho_ext_6, axis=(2))[180//2,:].real, label = 'neutral defect state')

#plt.plot(np.sum(rho_tot-1*rho_ind, axis=(2))[ttkw.fft_nx//2,:].real, label = r'$\rho_{ext}$')
#plt.plot(np.sum(rho_tot, axis=(2))[0*ttkw.fft_nx//2,:].real, label = r'$\rho_{tot}$')
#plt.ylim(-0.01,0.01)
#plt.legend()


In [ ]:
plt.plot(np.sum(1*(rho_B4210+rho_B4211), axis=(2))[540//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(1*(rho_B2923+rho_B2924), axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(0.5*(rho_B1870+rho_B1871), axis=(2))[360//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(0.5*(rho_B1051+rho_B1052), axis=(2))[0*ttkw.fft_nx//2,:].real, label = 'neutral defect state')
plt.plot(np.sum(0.5*(rho_B466+rho_B467), axis=(2))[180//2,:].real, label = 'neutral defect state')

In [ ]:
plt.plot(0.5*(rho_B4210+rho_B4211)[540//2,:,225//2])
plt.plot(0.5*(rho_B1870+rho_B1871)[360//2,:,225//2])

In [ ]:
plt.plot((pot_d-pot_p)[:,90,225//2+15])

In [ ]:
plt.plot(np.sum((pot_d-pot_p)[:,:,:], axis=(0,1))*ttkw.lattpara[0]*ttkw.lattpara[1]/180/180*np.sqrt(3)/2)

In [ ]:
dpot = pot_d-pot_p



In [ ]:

delta_pot = (pot_d-pot_p)[:,:,225//2]


delta_pot2 = np.zeros_like(delta_pot)

nx = ttkw.fft_nx
delta_pot2[:nx//2,:nx//2]=delta_pot[nx//2:,nx//2:]
delta_pot2[:nx//2,-nx//2:]=delta_pot[nx//2:,:nx//2]
delta_pot2[-nx//2:,:nx//2]=delta_pot[:nx//2,-nx//2:]
delta_pot2[-nx//2:,-nx//2:]=delta_pot[:nx//2,:nx//2]

#rho_ext_9 = rho_bare
pot_k = np.fft.fftn(delta_pot2)*ttkw.lattpara[0]*ttkw.lattpara[1]*np.sqrt(3)/2/180/180

X = (ttkw.fft_kxx[:,:,ttkw.fft_nz//2]).flatten()
Y = (ttkw.fft_kyy[:,:,ttkw.fft_nz//2]*2*np.sqrt(3)/3 + np.sqrt(3)/3* ttkw.fft_kxx[:,:,ttkw.fft_nz//2]).flatten()
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(X, Y, c=pot_k.flatten().real, s=30,marker='H',
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(-7, 7)
ax.set_ylim(-7, 7)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()